# Proof of Concept: CSLR via NLP (Continuous Sign Language Recognition)

Questo notebook dimostra l'approccio **Matrix of Solutions** (N-Best Rescoring) per superare i limiti della visione pura.
L'obiettivo è elevare l'Accuracy Top-1 sfruttando la coerenza grammaticale di un Language Model (**DistilBERT**).


In [ ]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

print('Caricamento NLP DistilBERT...')
nlp_corrector = pipeline('fill-mask', model='distilbert-base-uncased')

print('Caricamento Vocabolario ASL...')
checkpoint = torch.load('../models_saved/best_model.pt', map_location='cpu')
vocab = [w.lower() for w in checkpoint['class_to_idx'].keys() if w.isalpha() and len(w) > 2]
print(f'Vocabolario caricato: {len(vocab)} parole valide.')

## 1. Generazione della Matrice delle Soluzioni Visive
Simuliamo l'output della LSTM per **2000 frasi continue** come una **griglia di candidati**.

In [ ]:
nouns = [w for w in vocab if w in ['apple', 'book', 'house', 'water', 'car', 'computer', 'school', 'teacher', 'dog', 'family']]
verbs = [w for w in vocab if w in ['eat', 'drink', 'go', 'work', 'sleep', 'help', 'know', 'think', 'read']]
adjs = [w for w in vocab if w in ['good', 'bad', 'happy', 'sad', 'big', 'small', 'hot', 'cold']]

if not nouns: nouns = ['apple', 'house', 'water']
if not verbs: verbs = ['eat', 'go']
if not adjs: adjs = ['good', 'bad']

templates = [('i', 'want', 'noun'), ('he', 'verb', 'noun'), ('they', 'are', 'adj'), ('the', 'noun', 'is', 'adj'), ('please', 'verb', 'the', 'noun')]

synthetic_phrases = []
for _ in range(2000):
    t = random.choice(templates)
    p = [random.choice(nouns) if x == 'noun' else random.choice(verbs) if x == 'verb' else random.choice(adjs) if x == 'adj' else x for x in t]
    synthetic_phrases.append(p)

lstm_simulated_outputs = []
for phrase in synthetic_phrases:
    phrase_output = []
    for word in phrase:
        if random.random() < 0.54:
            top5 = [word] + random.sample([v for v in vocab if v != word], 4)
        else:
            top5 = random.sample([v for v in vocab if v != word], 5)
            top5[random.randint(1, 4)] = word
        phrase_output.append({'true_word': word, 'top5': top5})
    lstm_simulated_outputs.append(phrase_output)

print(f"Generate {len(lstm_simulated_outputs)} Matrici di Soluzioni Visive.")

## 2. Implementazione Matrix of Solutions (2-Pass Rescoring)
L'NLP valuta la matrice in due passaggi per eliminare il rumore visivo residuo.

In [ ]:
def matrix_rescoring(phrase_data, passes=2):
    current_best_phrase = [w["top5"][0] for w in phrase_data]
    
    for p in range(passes):
        new_best_phrase = []
        for i, target_data in enumerate(phrase_data):
            candidates = target_data["top5"]
            context = current_best_phrase.copy()
            context[i] = nlp_corrector.tokenizer.mask_token
            sentence = " ".join(context)
            
            try:
                # BERT rescoring su matrice visiva N-best
                preds = nlp_corrector(sentence, top_k=150)
                bert_scores = {pr["token_str"].strip().lower(): pr["score"] for pr in preds}
                
                best_word = candidates[0]
                best_score = -1.0
                for c in candidates:
                    s = bert_scores.get(c, 0.0)
                    if c == candidates[0]: s += 0.005 # Bias visivo minimo
                    if s > best_score:
                        best_score = s
                        best_word = c
                new_best_phrase.append(best_word)
            except:
                new_best_phrase.append(candidates[0])
        current_best_phrase = new_best_phrase
    return current_best_phrase

print("Algoritmo Matrix Rescoring pronto.")

## 3. Valutazione e Visualizzazione Risultati
Eseguiamo la pipeline e generiamo grafici per evidenziare il miglioramento dell'accuratezza.

In [ ]:
import matplotlib.pyplot as plt
correct_v = 0; correct_n = 0; total = 0
example_count = 0

for phrase_data in tqdm(lstm_simulated_outputs):
    true_p = [w["true_word"] for w in phrase_data]
    raw_p = [w["top5"][0] for w in phrase_data]
    corr_p = matrix_rescoring(phrase_data, passes=2)
    
    for r, v, n in zip(true_p, raw_p, corr_p):
        total += 1
        if v == r: correct_v += 1
        if n == r: correct_n += 1
    
    if example_count < 3 and raw_p != true_p and corr_p == true_p:
        print(f"\n[WIN] Reale: {' '.join(true_p).upper()} | Visione: {' '.join(raw_p).upper()} | Matrix NLP: {' '.join(corr_p).upper()}")
        example_count += 1

acc_v = (correct_v / total) * 100
acc_n = (correct_n / total) * 100

print("\n" + "="*60)
print("RISULTATI FINALI: MATRICE DELLE SOLUZIONI")
print("="*60)
print(f"Accuracy Visione (Top-1): {acc_v:.2f}%")
print(f"Accuracy Matrix NLP:     {acc_n:.2f}%")
print(f"Guadagno Netto Assoluto: +{(acc_n-acc_v):.2f}%")
print("="*60)

# Generazione Grafico
labels = ['Solo Visione (Base)', 'Ibrido (NLP Matrix)']
values = [acc_v, acc_n]

plt.figure(figsize=(10, 6))
bars = plt.bar(labels, values, color=['#e74c3c', '#2ecc71'], alpha=0.8)
plt.ylim(0, 100)
plt.ylabel('Accuracy (%)')
plt.title('Miglioramento dell\'Accuratezza via Matrix of Solutions')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

  1%|          | 11/2000 [00:01<04:13,  7.84it/s]


[WIN] Reale: I WANT FAMILY | Visione: SNAPSTICK WANT FAMILY | Matrix NLP: I WANT FAMILY


  1%|          | 21/2000 [00:03<04:17,  7.69it/s]


[WIN] Reale: THE WATER IS SMALL | Visione: SKILL WATER IS SMALL | Matrix NLP: THE WATER IS SMALL


  1%|▏         | 29/2000 [00:04<03:43,  8.81it/s]


[WIN] Reale: PLEASE KNOW THE TEACHER | Visione: OVER KNOW THE TEACHER | Matrix NLP: PLEASE KNOW THE TEACHER


 17%|█▋        | 333/2000 [00:38<03:13,  8.64it/s]